
# GPT-1 Decoder-only 챗봇 사전학습 QUEST

이 노트북은 기존 **Encoder–Decoder Transformer 챗봇**을 GPT-1 방식의 **Decoder-only 생성 모델**로 다시 구성한 제출본입니다.

> 제출 전 권장 순서  
> **Kernel → Restart & Clear Output → Run All → 결과가 출력된 상태로 저장**

---

## 1. Transformer에서 GPT-1로 바꾸는 핵심

### 기존 Transformer

```text
질문 토큰
   ↓
Encoder Self-Attention
   ↓
Encoder Memory
   ↓
Decoder Masked Self-Attention
   ↓
Cross-Attention
   ↓
답변 토큰
```

### GPT-1

```text
질문과 답변을 하나의 토큰열로 연결
   ↓
Token Embedding + Learned Position Embedding
   ↓
Causal Masked Self-Attention + FFN × N
   ↓
Vocabulary Logits
   ↓
다음 토큰 생성
```

### 구조 변경표

| 항목 | 기존 Transformer | GPT-1 제출 구현 |
|---|---|---|
| 전체 구조 | Encoder + Decoder | **Decoder-only** |
| Encoder | 사용 | **제거** |
| Cross-Attention | 사용 | **제거** |
| Self-Attention | Encoder/Decoder에 각각 존재 | **Causal Self-Attention만 사용** |
| 입력 | 질문과 답변을 분리 | **질문과 답변을 하나의 토큰열로 직렬화** |
| 위치 정보 | 정현파 위치 인코딩 가능 | **학습 가능한 위치 임베딩** |
| 학습 목표 | 답변 문장 예측 | **모든 위치의 다음 토큰 예측** |
| 과제 범위 | 지도학습·검색 fallback 가능 | **사전학습 데이터와 학습만 사용** |

코드에서 중요한 변경 지점은 다음 주석으로 표시합니다.

- `[QUEST 1]` Transformer에서 GPT-1로 바꾼 부분
- `[QUEST 2]` Decoder 생성 모델용 전처리
- `[QUEST 3]` 위치 정보 추가
- `[QUEST 4]` GPT-1 모델과 학습
- `[QUEST 5]` 입력에 따른 출력 생성

---

## 평가 기준 대응

1. 이 첫 Markdown에서 변경할 구조를 글과 블록으로 설명합니다.
2. 챗봇 Q/A 데이터를 하나의 causal language-model 토큰열로 바꿉니다.
3. 토큰 임베딩에 학습 가능한 위치 임베딩을 더합니다.
4. GPT-1 decoder-only 모델을 출력하고 학습 진행을 기록합니다.
5. 질문을 입력해 모델이 답변 토큰을 생성하는지 확인합니다.



## 2. 실행 환경 준비

필요한 패키지를 확인하고, 빠진 패키지만 조용히 설치합니다.  
준비 운동이 길면 모델도 지치니 필요한 것만 챙깁니다.


In [1]:

# 파이썬 실행 정보를 가져옵니다.
import sys

# 패키지 설치 명령을 실행할 도구입니다.
import subprocess

# 패키지가 이미 있는지 살펴볼 탐정입니다.
import importlib.util

# 필요한 모듈 이름과 설치 이름을 짝지어 둡니다.
REQUIRED_PACKAGES = {
    "torch": "torch",
    "pandas": "pandas",
    "requests": "requests",
    "tokenizers": "tokenizers",
    "tqdm": "tqdm",
    "matplotlib": "matplotlib",
}

# 아직 설치되지 않은 패키지만 골라냅니다.
missing_packages = [
    pip_name
    for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

# 빠진 패키지가 있을 때만 설치 열차를 출발시킵니다.
if missing_packages:
    # 무엇을 설치하는지 먼저 알려 줍니다.
    print("설치할 패키지:", missing_packages)

    # 현재 파이썬 환경에 패키지를 설치합니다.
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *missing_packages,
        ]
    )

# 이미 모두 준비됐다면 반갑게 통과합니다.
else:
    # 환경 준비가 끝났음을 보여 줍니다.
    print("필수 패키지가 이미 준비되어 있습니다.")


필수 패키지가 이미 준비되어 있습니다.



## 3. 라이브러리와 재현성 설정

랜덤 시드를 고정하고 GPU 사용 가능 여부를 확인합니다.  
같은 씨앗에서 비슷한 결과가 자라도록 화분을 정돈하는 단계입니다.


In [2]:

# 미래 타입 힌트를 편하게 사용합니다.
from __future__ import annotations

# 수학 계산을 맡깁니다.
import math

# 파이썬 랜덤을 고정할 때 사용합니다.
import random

# 문장을 정리할 정규식을 불러옵니다.
import re

# 설정값을 보기 좋게 묶습니다.
from dataclasses import dataclass, asdict

# 파일 경로를 운영체제와 친하게 다룹니다.
from pathlib import Path

# 선택형 타입을 읽기 쉽게 표시합니다.
from typing import Optional

# 공개 데이터를 내려받을 때 사용합니다.
import requests

# 표 형태 데이터를 다룹니다.
import pandas as pd

# 딥러닝의 본체를 불러옵니다.
import torch

# 신경망 부품 상자를 엽니다.
import torch.nn as nn

# 활성화 함수와 loss 함수를 사용합니다.
import torch.nn.functional as F

# 데이터셋과 배치 로더를 불러옵니다.
from torch.utils.data import Dataset, DataLoader

# Byte-level BPE 토크나이저 부품을 가져옵니다.
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

# 학습 진행률을 예쁘게 보여 줍니다.
from tqdm.auto import tqdm

# 학습 곡선을 그립니다.
import matplotlib.pyplot as plt

# 실험을 다시 재현할 씨앗입니다.
SEED = 42

# 랜덤 상태를 한 번에 고정하는 함수입니다.
def set_seed(seed: int = SEED) -> None:
    # 파이썬 랜덤을 고정합니다.
    random.seed(seed)

    # CPU용 PyTorch 랜덤을 고정합니다.
    torch.manual_seed(seed)

    # 모든 GPU의 랜덤을 고정합니다.
    torch.cuda.manual_seed_all(seed)

    # 가능한 범위에서 같은 연산 순서를 사용합니다.
    torch.backends.cudnn.deterministic = True

    # 속도 탐색 대신 재현성을 우선합니다.
    torch.backends.cudnn.benchmark = False

# 랜덤 씨앗을 심습니다.
set_seed()

# GPU가 있으면 GPU를, 없으면 CPU를 선택합니다.
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# 실제 학습 장치를 확인합니다.
print("사용 장치:", DEVICE)

# PyTorch 버전도 기록해 둡니다.
print("PyTorch 버전:", torch.__version__)


사용 장치: cpu
PyTorch 버전: 2.7.1+cpu



## 4. 챗봇 데이터 불러오기

로컬의 `ChatbotData.csv`를 먼저 찾고, 없으면 공개 저장소에서 내려받습니다.  
과제 범위는 **사전학습**이므로 감정 `label`은 사용하지 않고 `Q`, `A`만 사용합니다.


In [3]:

# 공개 한국어 챗봇 데이터 주소입니다.
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "songys/Chatbot_data/master/ChatbotData.csv"
)

# 결과물을 모아 둘 폴더입니다.
DATA_DIR = Path("./gpt1_quest_data")

# 폴더가 없으면 살짝 만들어 줍니다.
DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# 내려받은 CSV를 저장할 위치입니다.
DOWNLOADED_CSV_PATH = DATA_DIR / "ChatbotData.csv"

# 수업 환경에서 자주 쓰는 데이터 경로 후보입니다.
LOCAL_CANDIDATES = [
    Path("./ChatbotData.csv"),
    Path("./data/ChatbotData.csv"),
    Path("/content/ChatbotData.csv"),
    Path.home() / "aiffel" / "ChatbotData.csv",
    Path.home() / "aiffel" / "transformer_chatbot" / "data" / "ChatbotData.csv",
    DOWNLOADED_CSV_PATH,
]

# 인터넷이 막힌 환경에서 구조를 확인할 작은 구명보트입니다.
OFFLINE_EXAMPLES = [
    ("안녕", "안녕하세요. 반가워요."),
    ("오늘 기분이 어때?", "차분하고 좋아요."),
    ("공부가 잘 안돼", "작은 목표부터 천천히 시작해 보세요."),
    ("점심 메뉴를 추천해 줘", "따뜻한 국수나 비빔밥은 어때요?"),
    ("비가 와", "우산을 챙기고 미끄러운 길을 조심하세요."),
    ("주말에 뭐 할까?", "산책하거나 보고 싶던 영화를 보는 건 어때요?"),
    ("너는 누구야?", "대화를 배우고 있는 작은 GPT 모델이에요."),
    ("고마워", "천만에요. 도움이 되어 기뻐요."),
    ("피곤해", "오늘은 조금 쉬어 가도 괜찮아요."),
    ("기분이 좋아", "좋은 기분을 오래 즐겨 보세요."),
    ("배가 고파", "가볍고 맛있는 음식을 챙겨 드세요."),
    ("잠이 안 와", "조명을 낮추고 천천히 호흡해 보세요."),
]

# 데이터 파일을 찾거나 내려받는 함수입니다.
def load_chatbot_data() -> tuple[pd.DataFrame, str]:
    # 로컬 후보 경로를 하나씩 확인합니다.
    for candidate in LOCAL_CANDIDATES:
        # 파일이 실제로 있으면 바로 사용합니다.
        if candidate.exists():
            # 발견한 위치를 알려 줍니다.
            print("로컬 데이터 사용:", candidate)

            # CSV를 표로 읽습니다.
            frame = pd.read_csv(candidate)

            # 데이터 출처도 함께 돌려줍니다.
            return frame, str(candidate)

    # 로컬에 없으면 공개 데이터를 내려받습니다.
    try:
        # 네트워크 요청을 보냅니다.
        response = requests.get(
            DATA_URL,
            timeout=30,
        )

        # 오류 응답이면 즉시 알려 줍니다.
        response.raise_for_status()

        # 받은 파일을 디스크에 저장합니다.
        DOWNLOADED_CSV_PATH.write_bytes(
            response.content
        )

        # 다운로드 성공을 기록합니다.
        print("데이터 다운로드 완료:", DOWNLOADED_CSV_PATH)

        # 저장한 CSV를 읽습니다.
        frame = pd.read_csv(
            DOWNLOADED_CSV_PATH
        )

        # 데이터와 출처를 반환합니다.
        return frame, str(DOWNLOADED_CSV_PATH)

    # 네트워크가 막혀도 노트북 구조는 확인할 수 있습니다.
    except Exception as error:
        # fallback 사용 사실을 분명히 알립니다.
        print("[경고] 공개 데이터 다운로드 실패:", repr(error))

        # 작은 예시 데이터를 표로 만듭니다.
        frame = pd.DataFrame(
            OFFLINE_EXAMPLES,
            columns=["Q", "A"],
        )

        # fallback 출처를 표시합니다.
        return frame, "OFFLINE_EXAMPLES"

# 실제 데이터를 불러옵니다.
raw_df, DATA_SOURCE = load_chatbot_data()

# Q와 A 열이 있는지 확인합니다.
if not {"Q", "A"}.issubset(raw_df.columns):
    # 필요한 열이 없으면 친절하게 중단합니다.
    raise ValueError(
        "데이터에는 Q와 A 열이 필요합니다. "
        f"현재 열: {raw_df.columns.tolist()}"
    )

# 과제는 pre-training만 다루므로 Q와 A만 남깁니다.
raw_df = raw_df[["Q", "A"]].copy()

# 불러온 데이터의 크기를 확인합니다.
print("데이터 출처:", DATA_SOURCE)

# 전체 대화 수를 확인합니다.
print("원본 대화 수:", len(raw_df))

# 앞부분을 눈으로 살펴봅니다.
display(raw_df.head())


데이터 다운로드 완료: gpt1_quest_data/ChatbotData.csv
데이터 출처: gpt1_quest_data/ChatbotData.csv
원본 대화 수: 11823


,Q,A
0,12시 땡!,하루가 또 가네요.
1,1지망 학교 떨어졌어,위로해 드립니다.
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.
4,PPL 심하네,눈살이 찌푸려지죠.



## 5. Decoder 생성 모델용 전처리

GPT-1에는 질문과 답변을 따로 넣지 않습니다.  
한 대화를 다음 형식의 **하나의 토큰열**로 바꿉니다.

```text
<bos><speaker1>질문<turn><speaker2>답변<eos>
```

이렇게 하면 모델은 앞 토큰을 보고 다음 토큰을 예측하는 사전학습을 수행할 수 있습니다.


In [4]:

# 모델이 대화 경계를 알아볼 특수 토큰입니다.
SPECIAL_TOKENS = [
    "<pad>",
    "<unk>",
    "<bos>",
    "<eos>",
    "<speaker1>",
    "<speaker2>",
    "<turn>",
]

# 문장을 가볍게 청소하는 함수입니다.
def clean_text(text: str) -> str:
    # 결측치는 빈 문자열로 바꿉니다.
    text = (
        ""
        if pd.isna(text)
        else str(text)
    )

    # HTML 태그를 걷어 냅니다.
    text = re.sub(
        r"<[^>]+>",
        " ",
        text,
    )

    # URL은 대화 학습에 방해되지 않게 제거합니다.
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text,
    )

    # 제어 문자를 조용히 치웁니다.
    text = re.sub(
        r"[\x00-\x1f\x7f]",
        " ",
        text,
    )

    # 반복된 공백을 한 칸으로 정리합니다.
    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    # 양끝 공백을 다듬어 반환합니다.
    return text.strip()

# 질문과 답변을 GPT용 한 줄 대화로 묶습니다.
def serialize_dialogue(
    question: str,
    answer: str,
) -> str:
    # 질문을 깨끗하게 닦습니다.
    question = clean_text(question)

    # 답변도 같은 수건으로 닦습니다.
    answer = clean_text(answer)

    # [QUEST 2] Q와 A를 하나의 causal-LM 시퀀스로 연결합니다.
    return (
        f"<bos>"
        f"<speaker1>{question}"
        f"<turn>"
        f"<speaker2>{answer}"
        f"<eos>"
    )

# 원본을 보존하며 정제용 복사본을 만듭니다.
clean_df = raw_df.copy()

# 질문 문장을 정리합니다.
clean_df["Q"] = clean_df["Q"].map(
    clean_text
)

# 답변 문장도 정리합니다.
clean_df["A"] = clean_df["A"].map(
    clean_text
)

# 빈 질문과 빈 답변은 학습 무대에서 내려옵니다.
clean_df = clean_df[
    (clean_df["Q"].str.len() > 0)
    & (clean_df["A"].str.len() > 0)
].copy()

# 지나치게 긴 문장은 작은 실습 모델을 위해 제외합니다.
clean_df = clean_df[
    (clean_df["Q"].str.len() <= 200)
    & (clean_df["A"].str.len() <= 200)
].copy()

# 같은 질문과 답변의 복제본을 제거합니다.
clean_df = clean_df.drop_duplicates(
    subset=["Q", "A"]
)

# 인덱스를 단정하게 다시 붙입니다.
clean_df = clean_df.reset_index(
    drop=True
)

# 각 대화를 GPT-1 입력 문자열로 직렬화합니다.
clean_df["serialized_text"] = [
    serialize_dialogue(
        question,
        answer,
    )
    for question, answer in zip(
        clean_df["Q"],
        clean_df["A"],
    )
]

# 정제 결과를 확인합니다.
print("정제 후 대화 수:", len(clean_df))

# 실제 변환 예시를 보여 줍니다.
print("직렬화 예시:")
print(clean_df.loc[0, "serialized_text"])

# 첫 세 행을 표로 확인합니다.
display(
    clean_df[
        ["Q", "A", "serialized_text"]
    ].head(3)
)


정제 후 대화 수: 11750
직렬화 예시:
<bos><speaker1>12시 땡!<turn><speaker2>하루가 또 가네요.<eos>


,Q,A,serialized_text
0,12시 땡!,하루가 또 가네요.,<bos><speaker1>12시 땡!<turn><speaker2>하루가 또 가네요...
1,1지망 학교 떨어졌어,위로해 드립니다.,<bos><speaker1>1지망 학교 떨어졌어<turn><speaker2>위로해 ...
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,<bos><speaker1>3박4일 놀러가고 싶다<turn><speaker2>여행은...



## 6. 학습·검증 데이터 분리

토큰을 모두 합친 뒤 자르지 않고, **대화 단위로 먼저 분리**합니다.  
한 대화가 학습과 검증 양쪽에 걸치는 데이터 누수를 막는 작은 방파제입니다.


In [5]:

# 전체 대화를 같은 씨앗으로 섞습니다.
shuffled_df = clean_df.sample(
    frac=1.0,
    random_state=SEED,
).reset_index(
    drop=True
)

# 학습 데이터 비율입니다.
TRAIN_RATIO = 0.9

# 학습과 검증의 경계 위치를 계산합니다.
split_index = int(
    len(shuffled_df)
    * TRAIN_RATIO
)

# 데이터가 아주 작아도 양쪽에 최소 한 행은 남깁니다.
split_index = max(
    1,
    min(
        split_index,
        len(shuffled_df) - 1,
    ),
)

# 앞 90%를 학습 데이터로 사용합니다.
train_df = shuffled_df.iloc[
    :split_index
].copy()

# 뒤 10%를 검증 데이터로 사용합니다.
valid_df = shuffled_df.iloc[
    split_index:
].copy()

# 학습 대화 수를 확인합니다.
print("학습 대화 수:", len(train_df))

# 검증 대화 수를 확인합니다.
print("검증 대화 수:", len(valid_df))

# 두 데이터가 겹치지 않는지 질문·답변 쌍으로 검사합니다.
train_pairs = set(
    zip(
        train_df["Q"],
        train_df["A"],
    )
)

# 검증 쌍도 같은 방식으로 모읍니다.
valid_pairs = set(
    zip(
        valid_df["Q"],
        valid_df["A"],
    )
)

# 교집합이 비어 있어야 데이터 누수가 없습니다.
overlap_count = len(
    train_pairs
    & valid_pairs
)

# 누수 검사 결과를 출력합니다.
print("train/valid 중복 대화 수:", overlap_count)

# 중복이 있으면 즉시 알려 줍니다.
assert overlap_count == 0


학습 대화 수: 10575
검증 대화 수: 1175
train/valid 중복 대화 수: 0



## 7. Byte-level BPE 토크나이저 학습

GPT-1의 BPE 아이디어를 따라 Byte-level BPE를 학습합니다.  
검증 문장을 미리 엿보지 않도록 토크나이저는 **학습 데이터만** 사용합니다.


In [6]:

# 실습 데이터에 맞춘 목표 어휘 크기입니다.
TOKENIZER_VOCAB_SIZE = 4_000

# 모르는 토큰을 처리하는 BPE 모델을 만듭니다.
tokenizer = Tokenizer(
    models.BPE(
        unk_token="<unk>"
    )
)

# 공백과 문자를 byte 수준에서 나눕니다.
tokenizer.pre_tokenizer = (
    pre_tokenizers.ByteLevel(
        add_prefix_space=False
    )
)

# 토큰 ID를 다시 읽을 수 있는 문장으로 복원합니다.
tokenizer.decoder = (
    decoders.ByteLevel()
)

# BPE 학습 규칙을 준비합니다.
tokenizer_trainer = trainers.BpeTrainer(
    # 목표 어휘 수를 지정합니다.
    vocab_size=TOKENIZER_VOCAB_SIZE,

    # 한 번만 나온 조각은 지나치게 외우지 않게 합니다.
    min_frequency=2,

    # 대화 경계 토큰을 특별 대우합니다.
    special_tokens=SPECIAL_TOKENS,

    # 새로운 문자도 처리하도록 byte 알파벳을 준비합니다.
    initial_alphabet=(
        pre_tokenizers
        .ByteLevel
        .alphabet()
    ),

    # 학습 진행률을 보여 줍니다.
    show_progress=True,
)

# 학습 데이터만 사용해 토크나이저를 학습합니다.
tokenizer.train_from_iterator(
    train_df["serialized_text"].tolist(),
    trainer=tokenizer_trainer,
)

# 토크나이저 저장 경로를 정합니다.
TOKENIZER_PATH = (
    DATA_DIR
    / "gpt1_chatbot_tokenizer.json"
)

# 다시 사용할 수 있도록 파일로 저장합니다.
tokenizer.save(
    str(TOKENIZER_PATH)
)

# 실제 생성된 어휘 크기를 확인합니다.
VOCAB_SIZE = tokenizer.get_vocab_size()

# PAD 토큰 번호를 꺼냅니다.
PAD_ID = tokenizer.token_to_id("<pad>")

# UNK 토큰 번호를 꺼냅니다.
UNK_ID = tokenizer.token_to_id("<unk>")

# BOS 토큰 번호를 꺼냅니다.
BOS_ID = tokenizer.token_to_id("<bos>")

# EOS 토큰 번호를 꺼냅니다.
EOS_ID = tokenizer.token_to_id("<eos>")

# 화자 1 토큰 번호를 꺼냅니다.
SPEAKER1_ID = tokenizer.token_to_id("<speaker1>")

# 화자 2 토큰 번호를 꺼냅니다.
SPEAKER2_ID = tokenizer.token_to_id("<speaker2>")

# 질문과 답변 사이 토큰 번호를 꺼냅니다.
TURN_ID = tokenizer.token_to_id("<turn>")

# 실제 어휘 크기를 출력합니다.
print("실제 어휘 크기:", VOCAB_SIZE)

# 특수 토큰 번호표를 확인합니다.
print(
    "특수 토큰 ID:",
    {
        token: tokenizer.token_to_id(token)
        for token in SPECIAL_TOKENS
    },
)

# 첫 번째 학습 문장을 토큰으로 바꿉니다.
sample_encoding = tokenizer.encode(
    train_df.iloc[0]["serialized_text"]
)

# 토큰 ID 일부를 보여 줍니다.
print(
    "샘플 token IDs:",
    sample_encoding.ids[:30],
)

# 다시 문장으로 복원해 토큰화가 정상인지 확인합니다.
print(
    "복원 문장:",
    tokenizer.decode(
        sample_encoding.ids,
        skip_special_tokens=False,
    ),
)





실제 어휘 크기: 4000
특수 토큰 ID: {'<pad>': 0, '<unk>': 1, '<bos>': 2, '<eos>': 3, '<speaker1>': 4, '<speaker2>': 5, '<turn>': 6}
샘플 token IDs: [2, 4, 1362, 1684, 1852, 37, 6, 5, 458, 964, 1268, 284, 874, 20, 3]
복원 문장: <bos><speaker1>사랑이란 뭘까?<turn><speaker2>사랑에는 답이 없어요.<eos>



## 8. GPT-1 설정

논문 규모와 실습 규모를 모두 정의합니다.

- `paper`: GPT-1의 주요 규모인 12층, 768차원, 12개 head, 512 문맥
- `practice`: 같은 decoder-only 구조를 유지하면서 수업 환경에서 실행하기 쉽게 폭과 문맥을 줄인 버전

기본값은 `practice`입니다. GPU 메모리와 시간이 충분하면 `MODEL_PRESET = "paper"`로 바꿀 수 있습니다.


In [7]:

# GPT-1 설정을 한 상자에 담습니다.
@dataclass
class GPT1Config:
    # 토크나이저 어휘 크기입니다.
    vocab_size: int

    # 모델이 한 번에 보는 최대 토큰 수입니다.
    block_size: int

    # Decoder 블록 개수입니다.
    n_layer: int

    # Multi-Head Attention의 head 개수입니다.
    n_head: int

    # 각 토큰의 hidden 차원입니다.
    n_embd: int

    # Feed-Forward 중간 차원입니다.
    n_ff: int

    # 과적합을 줄일 dropout 비율입니다.
    dropout: float = 0.1

    # Linear 층의 bias 사용 여부입니다.
    bias: bool = True

# 설정 이름에 맞는 GPT-1 구성을 만드는 함수입니다.
def make_gpt1_config(
    preset: str,
    vocab_size: int,
) -> GPT1Config:
    # 논문 규모를 선택한 경우입니다.
    if preset == "paper":
        # GPT-1 논문의 주요 모델 규모를 반환합니다.
        return GPT1Config(
            vocab_size=vocab_size,
            block_size=512,
            n_layer=12,
            n_head=12,
            n_embd=768,
            n_ff=3072,
            dropout=0.1,
        )

    # 실습 규모도 12개 Decoder 블록은 유지합니다.
    return GPT1Config(
        vocab_size=vocab_size,
        block_size=128,
        n_layer=12,
        n_head=6,
        n_embd=384,
        n_ff=1536,
        dropout=0.1,
    )

# 실제 실행할 모델 규모를 선택합니다.
MODEL_PRESET = "practice"

# 실제 학습 설정을 만듭니다.
config = make_gpt1_config(
    MODEL_PRESET,
    VOCAB_SIZE,
)

# 논문 설정도 비교용으로 만듭니다.
paper_config = make_gpt1_config(
    "paper",
    VOCAB_SIZE,
)

# 논문 사양을 출력합니다.
print("GPT-1 논문 규모:")
print(asdict(paper_config))

# 실제 실행 사양을 출력합니다.
print("\n실제 실행 규모:")
print(asdict(config))

# 축소 이유를 솔직하게 적습니다.
print(
    "\n실제 모델은 GPT-1의 12층 decoder-only 구조를 유지하고, "
    "실습 실행 시간을 위해 hidden 폭과 문맥 길이만 줄였습니다."
)


GPT-1 논문 규모:
{'vocab_size': 4000, 'block_size': 512, 'n_layer': 12, 'n_head': 12, 'n_embd': 768, 'n_ff': 3072, 'dropout': 0.1, 'bias': True}

실제 실행 규모:
{'vocab_size': 4000, 'block_size': 128, 'n_layer': 12, 'n_head': 6, 'n_embd': 384, 'n_ff': 1536, 'dropout': 0.1, 'bias': True}

실제 모델은 GPT-1의 12층 decoder-only 구조를 유지하고, 실습 실행 시간을 위해 hidden 폭과 문맥 길이만 줄였습니다.



## 9. Causal Language Model 데이터셋

각 질문·답변 쌍을 독립된 샘플로 유지합니다.

```text
전체 토큰: [t0, t1, t2, ..., tN]
입력:      [t0, t1, t2, ..., tN-1]
정답:      [t1, t2, t3, ..., tN]
```

부족한 길이는 `<pad>`로 채우고, padding 정답은 `-100`으로 표시해 loss 계산에서 제외합니다.  
`position_ids`도 이 단계에서 `0, 1, 2, ...`로 만들어 위치 정보를 데이터에 붙입니다.


In [8]:

# Q/A 한 쌍을 GPT-1 학습 샘플로 바꾸는 데이터셋입니다.
class GPT1PretrainDataset(Dataset):
    # 데이터와 토크나이저를 저장합니다.
    def __init__(
        self,
        frame: pd.DataFrame,
        tokenizer: Tokenizer,
        block_size: int,
    ):
        # 행 번호를 깔끔하게 다시 맞춥니다.
        self.frame = frame.reset_index(
            drop=True
        )

        # 토크나이저를 보관합니다.
        self.tokenizer = tokenizer

        # 최대 입력 길이를 기억합니다.
        self.block_size = block_size

    # 전체 샘플 수를 알려 줍니다.
    def __len__(self) -> int:
        # 대화 한 행이 샘플 하나입니다.
        return len(self.frame)

    # 특정 대화를 모델 입력으로 바꿉니다.
    def __getitem__(
        self,
        index: int,
    ) -> dict[str, torch.Tensor]:
        # 직렬화된 대화를 가져옵니다.
        text = self.frame.iloc[
            index
        ]["serialized_text"]

        # 문장을 토큰 ID로 바꿉니다.
        token_ids = self.tokenizer.encode(
            text
        ).ids

        # 한 칸 이동용 토큰까지 포함해 자릅니다.
        token_ids = token_ids[
            : self.block_size + 1
        ]

        # 너무 짧은 예외에는 EOS를 덧붙입니다.
        if len(token_ids) < 2:
            # 최소한 시작과 끝은 남겨 둡니다.
            token_ids = [
                BOS_ID,
                EOS_ID,
            ]

        # [QUEST 2] 마지막 토큰 전까지를 입력으로 사용합니다.
        input_ids = token_ids[:-1]

        # [QUEST 2] 첫 토큰 다음부터를 정답으로 사용합니다.
        labels = token_ids[1:]

        # 실제 토큰 위치에는 1을 표시합니다.
        attention_mask = [
            1
        ] * len(input_ids)

        # 고정 길이까지 필요한 padding 수를 계산합니다.
        pad_length = (
            self.block_size
            - len(input_ids)
        )

        # 입력 뒤에 PAD 토큰을 붙입니다.
        input_ids = input_ids + [
            PAD_ID
        ] * pad_length

        # padding 정답은 loss에서 빼기 위해 -100을 붙입니다.
        labels = labels + [
            -100
        ] * pad_length

        # padding 위치는 attention에서 숨깁니다.
        attention_mask = attention_mask + [
            0
        ] * pad_length

        # [QUEST 3] 각 토큰 자리에 위치 번호를 붙입니다.
        position_ids = list(
            range(self.block_size)
        )

        # 입력 토큰을 정수 텐서로 바꿉니다.
        input_ids_tensor = torch.tensor(
            input_ids,
            dtype=torch.long,
        )

        # 정답 토큰을 정수 텐서로 바꿉니다.
        labels_tensor = torch.tensor(
            labels,
            dtype=torch.long,
        )

        # attention mask도 정수 텐서로 바꿉니다.
        attention_mask_tensor = torch.tensor(
            attention_mask,
            dtype=torch.long,
        )

        # 위치 번호도 정수 텐서로 바꿉니다.
        position_ids_tensor = torch.tensor(
            position_ids,
            dtype=torch.long,
        )

        # 모델이 바로 먹을 수 있는 꾸러미로 반환합니다.
        return {
            "input_ids": input_ids_tensor,
            "position_ids": position_ids_tensor,
            "attention_mask": attention_mask_tensor,
            "labels": labels_tensor,
        }

# 학습용 데이터셋을 만듭니다.
train_dataset = GPT1PretrainDataset(
    train_df,
    tokenizer,
    config.block_size,
)

# 검증용 데이터셋을 만듭니다.
valid_dataset = GPT1PretrainDataset(
    valid_df,
    tokenizer,
    config.block_size,
)

# GPU에서는 조금 넉넉하게, CPU에서는 가볍게 갑니다.
BATCH_SIZE = (
    8
    if DEVICE.type == "cuda"
    else 2
)

# 학습 데이터를 섞어서 배치로 만듭니다.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(
        DEVICE.type == "cuda"
    ),
)

# 검증 데이터는 순서를 유지합니다.
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(
        DEVICE.type == "cuda"
    ),
)

# 첫 배치를 꺼내 모양을 살펴봅니다.
sample_batch = next(
    iter(train_loader)
)

# 각 텐서의 shape를 출력합니다.
for key, value in sample_batch.items():
    # 이름과 크기를 나란히 보여 줍니다.
    print(
        f"{key:15s}:",
        tuple(value.shape),
    )


input_ids      : (2, 128)
position_ids   : (2, 128)
attention_mask : (2, 128)
labels         : (2, 128)



## 10. GPT-1 Decoder-only 모델 구현

이 셀이 과제의 심장입니다.

- Encoder 없음
- Cross-Attention 없음
- 같은 hidden state에서 Q, K, V 생성
- 미래 토큰을 가리는 causal mask
- 학습 가능한 위치 임베딩
- GELU Feed-Forward
- Residual + LayerNorm
- 입력 임베딩과 출력 가중치 공유


In [9]:

# 미래 토큰을 보지 않는 Self-Attention입니다.
class CausalSelfAttention(nn.Module):
    # Attention 부품을 준비합니다.
    def __init__(
        self,
        config: GPT1Config,
    ):
        # nn.Module 초기화를 먼저 실행합니다.
        super().__init__()

        # embedding 차원이 head 수로 나뉘는지 확인합니다.
        if config.n_embd % config.n_head != 0:
            # 나머지가 생기면 head를 고르게 나눌 수 없습니다.
            raise ValueError(
                "n_embd는 n_head로 나누어 떨어져야 합니다."
            )

        # head 개수를 기억합니다.
        self.n_head = config.n_head

        # head 하나가 담당할 차원을 계산합니다.
        self.head_dim = (
            config.n_embd
            // config.n_head
        )

        # [QUEST 1] Encoder memory 없이 같은 x에서 Q, K, V를 만듭니다.
        self.qkv_proj = nn.Linear(
            config.n_embd,
            3 * config.n_embd,
            bias=config.bias,
        )

        # 여러 head의 결과를 다시 합칩니다.
        self.out_proj = nn.Linear(
            config.n_embd,
            config.n_embd,
            bias=config.bias,
        )

        # Attention 확률에 dropout을 적용합니다.
        self.attn_dropout = nn.Dropout(
            config.dropout
        )

        # 출력 residual에도 dropout을 적용합니다.
        self.resid_dropout = nn.Dropout(
            config.dropout
        )

        # [QUEST 1] 미래 칸에 커튼을 치는 하삼각 마스크입니다.
        causal_mask = torch.tril(
            torch.ones(
                config.block_size,
                config.block_size,
                dtype=torch.bool,
            )
        )

        # 마스크를 모델과 함께 장치로 이동하도록 등록합니다.
        self.register_buffer(
            "causal_mask",
            causal_mask.view(
                1,
                1,
                config.block_size,
                config.block_size,
            ),
            persistent=False,
        )

    # 한 번의 causal self-attention을 계산합니다.
    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        # 배치 크기, 길이, embedding 크기를 읽습니다.
        batch_size, seq_len, embed_dim = (
            hidden_states.shape
        )

        # Q, K, V를 한 번에 계산합니다.
        qkv = self.qkv_proj(
            hidden_states
        )

        # 마지막 차원을 Q, K, V 세 조각으로 나눕니다.
        query, key, value = qkv.chunk(
            3,
            dim=-1,
        )

        # Query를 여러 head 모양으로 바꿉니다.
        query = query.view(
            batch_size,
            seq_len,
            self.n_head,
            self.head_dim,
        ).transpose(
            1,
            2,
        )

        # Key도 같은 모양으로 바꿉니다.
        key = key.view(
            batch_size,
            seq_len,
            self.n_head,
            self.head_dim,
        ).transpose(
            1,
            2,
        )

        # Value도 head별 바구니에 담습니다.
        value = value.view(
            batch_size,
            seq_len,
            self.n_head,
            self.head_dim,
        ).transpose(
            1,
            2,
        )

        # 토큰 사이의 유사도 점수를 계산합니다.
        attention_scores = (
            query
            @ key.transpose(-2, -1)
        ) / math.sqrt(
            self.head_dim
        )

        # 현재 문장 길이에 맞는 causal mask를 꺼냅니다.
        causal_mask = self.causal_mask[
            :,
            :,
            :seq_len,
            :seq_len,
        ]

        # PAD key를 보지 않도록 mask 모양을 넓힙니다.
        key_padding_mask = attention_mask[
            :,
            None,
            None,
            :seq_len,
        ].bool()

        # 미래 토큰과 PAD 토큰을 동시에 가립니다.
        allowed_mask = (
            causal_mask
            & key_padding_mask
        )

        # 가려진 위치의 점수를 아주 작게 만듭니다.
        attention_scores = attention_scores.masked_fill(
            ~allowed_mask,
            torch.finfo(
                attention_scores.dtype
            ).min,
        )

        # 점수를 확률로 바꿉니다.
        attention_weights = F.softmax(
            attention_scores,
            dim=-1,
        )

        # Attention 확률에 살짝 dropout을 뿌립니다.
        attention_weights = self.attn_dropout(
            attention_weights
        )

        # 확률만큼 Value를 섞습니다.
        context = (
            attention_weights
            @ value
        )

        # 여러 head를 다시 한 줄로 모읍니다.
        context = context.transpose(
            1,
            2,
        ).contiguous().view(
            batch_size,
            seq_len,
            embed_dim,
        )

        # 최종 projection을 통과시킵니다.
        output = self.out_proj(
            context
        )

        # residual 길로 보내기 전에 dropout을 적용합니다.
        return self.resid_dropout(
            output
        )


# GPT-1의 position-wise Feed-Forward입니다.
class GPT1MLP(nn.Module):
    # 두 개의 Linear 층을 준비합니다.
    def __init__(
        self,
        config: GPT1Config,
    ):
        # 부모 클래스를 초기화합니다.
        super().__init__()

        # embedding을 넓은 공간으로 펼칩니다.
        self.fc = nn.Linear(
            config.n_embd,
            config.n_ff,
            bias=config.bias,
        )

        # 넓어진 표현을 다시 embedding 크기로 줄입니다.
        self.proj = nn.Linear(
            config.n_ff,
            config.n_embd,
            bias=config.bias,
        )

        # MLP 출력에도 dropout을 적용합니다.
        self.dropout = nn.Dropout(
            config.dropout
        )

    # MLP 계산을 실행합니다.
    def forward(
        self,
        hidden_states: torch.Tensor,
    ) -> torch.Tensor:
        # GPT-1에서 사용한 GELU로 비선형성을 더합니다.
        hidden_states = F.gelu(
            self.fc(hidden_states)
        )

        # 다시 원래 embedding 차원으로 돌아옵니다.
        hidden_states = self.proj(
            hidden_states
        )

        # 과적합을 줄인 출력을 반환합니다.
        return self.dropout(
            hidden_states
        )


# Attention과 MLP를 묶은 GPT-1 블록입니다.
class GPT1Block(nn.Module):
    # 한 블록의 부품을 조립합니다.
    def __init__(
        self,
        config: GPT1Config,
    ):
        # 부모 클래스를 초기화합니다.
        super().__init__()

        # causal self-attention을 준비합니다.
        self.attn = CausalSelfAttention(
            config
        )

        # Attention residual 뒤의 LayerNorm입니다.
        self.ln_1 = nn.LayerNorm(
            config.n_embd
        )

        # Feed-Forward 네트워크를 준비합니다.
        self.mlp = GPT1MLP(
            config
        )

        # MLP residual 뒤의 LayerNorm입니다.
        self.ln_2 = nn.LayerNorm(
            config.n_embd
        )

    # 한 Decoder 블록을 통과합니다.
    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        # [QUEST 1] Cross-Attention 없이 self-attention만 사용합니다.
        attention_output = self.attn(
            hidden_states,
            attention_mask,
        )

        # Residual을 더하고 정규화합니다.
        hidden_states = self.ln_1(
            hidden_states
            + attention_output
        )

        # 현재 표현을 MLP로 보냅니다.
        mlp_output = self.mlp(
            hidden_states
        )

        # 두 번째 residual과 정규화를 적용합니다.
        hidden_states = self.ln_2(
            hidden_states
            + mlp_output
        )

        # 다음 블록으로 넘깁니다.
        return hidden_states


# GPT-1 decoder-only 언어 모델입니다.
class GPT1LanguageModel(nn.Module):
    # 전체 GPT-1 모델을 조립합니다.
    def __init__(
        self,
        config: GPT1Config,
    ):
        # 부모 클래스를 초기화합니다.
        super().__init__()

        # 설정을 모델 안에 보관합니다.
        self.config = config

        # 토큰 ID를 의미 벡터로 바꿉니다.
        self.token_embedding = nn.Embedding(
            config.vocab_size,
            config.n_embd,
        )

        # [QUEST 3] 위치 ID를 학습 가능한 위치 벡터로 바꿉니다.
        self.position_embedding = nn.Embedding(
            config.block_size,
            config.n_embd,
        )

        # 입력 embedding 합에 dropout을 적용합니다.
        self.embedding_dropout = nn.Dropout(
            config.dropout
        )

        # [QUEST 1] Encoder 없이 GPT Decoder 블록만 반복합니다.
        self.blocks = nn.ModuleList(
            [
                GPT1Block(config)
                for _ in range(
                    config.n_layer
                )
            ]
        )

        # hidden state를 전체 어휘 점수로 바꿉니다.
        self.lm_head = nn.Linear(
            config.n_embd,
            config.vocab_size,
            bias=False,
        )

        # 모든 가중치를 GPT 방식으로 초기화합니다.
        self.apply(
            self._initialize_weights
        )

        # GPT-1 식으로 입력 embedding과 출력 weight를 공유합니다.
        self.lm_head.weight = (
            self.token_embedding.weight
        )

    # Linear와 Embedding의 초기값을 정합니다.
    def _initialize_weights(
        self,
        module: nn.Module,
    ) -> None:
        # Linear 층은 작은 정규분포에서 시작합니다.
        if isinstance(
            module,
            nn.Linear,
        ):
            # weight를 평균 0, 표준편차 0.02로 초기화합니다.
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02,
            )

            # bias가 있으면 0으로 시작합니다.
            if module.bias is not None:
                # bias를 깨끗한 0으로 맞춥니다.
                nn.init.zeros_(
                    module.bias
                )

        # Embedding도 같은 분포에서 시작합니다.
        elif isinstance(
            module,
            nn.Embedding,
        ):
            # 토큰과 위치 embedding을 초기화합니다.
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02,
            )

    # GPT-1의 forward 계산입니다.
    def forward(
        self,
        input_ids: torch.Tensor,
        position_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
    ) -> tuple[torch.Tensor, Optional[torch.Tensor]]:
        # 배치 크기와 시퀀스 길이를 확인합니다.
        batch_size, seq_len = (
            input_ids.shape
        )

        # 문맥 창보다 긴 입력은 받지 않습니다.
        if seq_len > self.config.block_size:
            # 길이 오류를 이해하기 쉽게 알려 줍니다.
            raise ValueError(
                f"입력 길이 {seq_len}이 "
                f"block_size {self.config.block_size}보다 큽니다."
            )

        # 토큰 번호를 토큰 embedding으로 바꿉니다.
        token_embeddings = self.token_embedding(
            input_ids
        )

        # [QUEST 3] 위치 번호를 위치 embedding으로 바꿉니다.
        position_embeddings = self.position_embedding(
            position_ids
        )

        # [QUEST 3] GPT 입력 블록에서 토큰과 위치 정보를 더합니다.
        hidden_states = (
            token_embeddings
            + position_embeddings
        )

        # 합쳐진 입력 표현에 dropout을 적용합니다.
        hidden_states = self.embedding_dropout(
            hidden_states
        )

        # 12개의 GPT 블록을 차례로 통과합니다.
        for block in self.blocks:
            # causal mask와 padding mask를 함께 전달합니다.
            hidden_states = block(
                hidden_states,
                attention_mask,
            )

        # 각 위치를 어휘 전체의 logits로 바꿉니다.
        logits = self.lm_head(
            hidden_states
        )

        # 정답이 없으면 추론 모드이므로 loss는 비워 둡니다.
        loss = None

        # 정답이 제공되면 다음 토큰 loss를 계산합니다.
        if labels is not None:
            # [QUEST 4] padding 정답 -100은 loss에서 제외합니다.
            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    logits.size(-1),
                ),
                labels.reshape(-1),
                ignore_index=-100,
            )

        # 생성 점수와 선택적 loss를 반환합니다.
        return logits, loss



## 11. 모델 출력과 구조 확인

`print(model)`로 실제 구성된 층을 확인합니다.  
Encoder나 Cross-Attention이 숨어 있지 않은지, Decoder 블록이 12개인지도 함께 검사합니다.


In [10]:

# 같은 시작점에서 모델을 만듭니다.
set_seed(SEED)

# 실제 GPT-1 모델을 장치에 올립니다.
model = GPT1LanguageModel(
    config
).to(
    DEVICE
)

# [QUEST 4] 모델 전체 구조를 출력합니다.
print(model)

# 전체 파라미터 수를 계산합니다.
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

# 학습 가능한 파라미터 수를 계산합니다.
trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

# 파라미터 수를 보기 좋게 출력합니다.
print(
    f"\n전체 파라미터 수: "
    f"{total_parameters:,}"
)

# 실제 학습 대상 수도 출력합니다.
print(
    f"학습 파라미터 수: "
    f"{trainable_parameters:,}"
)

# Decoder 블록 개수를 확인합니다.
print(
    "GPT 블록 수:",
    len(model.blocks),
)

# 입력과 출력 weight가 같은지 확인합니다.
print(
    "입력/출력 가중치 공유:",
    (
        model.token_embedding.weight.data_ptr()
        == model.lm_head.weight.data_ptr()
    ),
)

# 실습 모델도 12개 블록을 사용해야 합니다.
assert len(model.blocks) == 12


GPT1LanguageModel(
  (token_embedding): Embedding(4000, 384)
  (position_embedding): Embedding(128, 384)
  (embedding_dropout): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-11): 12 x GPT1Block(
      (attn): CausalSelfAttention(
        (qkv_proj): Linear(in_features=384, out_features=1152, bias=True)
        (out_proj): Linear(in_features=384, out_features=384, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT1MLP(
        (fc): Linear(in_features=384, out_features=1536, bias=True)
        (proj): Linear(in_features=1536, out_features=384, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
    )
  )
  (lm_head): Linear(in_features=384, out_features=4000, bias=False)
)

전체 파라미터 수: 22,878,720
학습 파라미터 수: 22,878,720
GPT 블록 수: 12



## 12. GPT 입력 블록과 위치 정보 검사

평가 기준 3을 직접 확인합니다.

- `input_ids`
- `position_ids`
- Token Embedding
- Position Embedding
- 두 임베딩의 합
- 최종 logits

각 shape가 자연스럽게 이어지는지 출력합니다.


In [11]:

# 첫 배치에서 두 샘플만 꺼냅니다.
test_batch = {
    key: value[:2].to(DEVICE)
    for key, value in sample_batch.items()
}

# 입력 토큰을 토큰 embedding으로 바꿉니다.
token_embedding_sample = model.token_embedding(
    test_batch["input_ids"]
)

# [QUEST 3] 위치 번호를 위치 embedding으로 바꿉니다.
position_embedding_sample = model.position_embedding(
    test_batch["position_ids"]
)

# 두 embedding을 더해 GPT 입력 표현을 만듭니다.
combined_embedding_sample = (
    token_embedding_sample
    + position_embedding_sample
)

# gradient 없이 한 번 forward를 실행합니다.
with torch.no_grad():
    # logits와 초기 loss를 계산합니다.
    test_logits, test_loss = model(
        input_ids=test_batch["input_ids"],
        position_ids=test_batch["position_ids"],
        attention_mask=test_batch["attention_mask"],
        labels=test_batch["labels"],
    )

# 입력 토큰 shape를 확인합니다.
print(
    "input_ids shape:",
    tuple(
        test_batch["input_ids"].shape
    ),
)

# 위치 정보 shape를 확인합니다.
print(
    "position_ids shape:",
    tuple(
        test_batch["position_ids"].shape
    ),
)

# 첫 샘플의 위치 번호 일부를 보여 줍니다.
print(
    "첫 position_ids:",
    test_batch[
        "position_ids"
    ][0, :20].tolist(),
)

# 토큰 embedding shape를 확인합니다.
print(
    "token embedding shape:",
    tuple(
        token_embedding_sample.shape
    ),
)

# 위치 embedding shape를 확인합니다.
print(
    "position embedding shape:",
    tuple(
        position_embedding_sample.shape
    ),
)

# 더해진 입력 표현 shape를 확인합니다.
print(
    "combined embedding shape:",
    tuple(
        combined_embedding_sample.shape
    ),
)

# vocabulary logits shape를 확인합니다.
print(
    "logits shape:",
    tuple(
        test_logits.shape
    ),
)

# 초기 loss가 정상 숫자인지 확인합니다.
print(
    "초기 loss:",
    float(test_loss),
)

# 토큰과 위치 embedding의 shape는 같아야 합니다.
assert (
    token_embedding_sample.shape
    == position_embedding_sample.shape
)

# logits의 마지막 차원은 어휘 크기여야 합니다.
assert (
    test_logits.size(-1)
    == config.vocab_size
)

# loss가 NaN이나 inf가 아닌지 확인합니다.
assert torch.isfinite(
    test_loss
)


input_ids shape: (2, 128)
position_ids shape: (2, 128)
첫 position_ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
token embedding shape: (2, 128, 384)
position embedding shape: (2, 128, 384)
combined embedding shape: (2, 128, 384)
logits shape: (2, 128, 4000)
초기 loss: 8.29974365234375



## 13. 미래 정보 누출 검사

같은 문장의 뒤쪽 토큰만 바꾼 뒤, 앞쪽 logits가 변하지 않는지 확인합니다.  
차이가 0에 가깝다면 causal mask가 미래를 잘 가리고 있다는 뜻입니다.


In [12]:

# 평가 모드로 바꿔 dropout을 잠시 멈춥니다.
model.eval()

# 첫 샘플을 복사합니다.
probe_input_ids = test_batch[
    "input_ids"
][:1].clone()

# 비교용 입력도 하나 더 복사합니다.
changed_input_ids = probe_input_ids.clone()

# 문장의 중간 지점을 찾습니다.
cut_position = max(
    2,
    probe_input_ids.size(1) // 2,
)

# 중간 이후의 미래 토큰만 무작위로 바꿉니다.
changed_input_ids[
    :,
    cut_position:,
] = torch.randint(
    low=0,
    high=config.vocab_size,
    size=changed_input_ids[
        :,
        cut_position:,
    ].shape,
    device=DEVICE,
)

# 원본과 변경본을 gradient 없이 비교합니다.
with torch.no_grad():
    # 원본 입력의 logits를 계산합니다.
    original_logits, _ = model(
        input_ids=probe_input_ids,
        position_ids=test_batch[
            "position_ids"
        ][:1],
        attention_mask=test_batch[
            "attention_mask"
        ][:1],
    )

    # 미래 토큰을 바꾼 입력의 logits를 계산합니다.
    changed_logits, _ = model(
        input_ids=changed_input_ids,
        position_ids=test_batch[
            "position_ids"
        ][:1],
        attention_mask=test_batch[
            "attention_mask"
        ][:1],
    )

# 미래가 바뀌기 전 prefix 차이의 최댓값을 계산합니다.
max_prefix_difference = (
    original_logits[
        :,
        :cut_position,
    ]
    - changed_logits[
        :,
        :cut_position,
    ]
).abs().max().item()

# 누출 검사 결과를 출력합니다.
print(
    "미래 토큰 변경 후 "
    "prefix logits 최대 차이:",
    max_prefix_difference,
)

# 차이가 거의 0이어야 causal mask가 정상입니다.
assert max_prefix_difference < 1e-5

# 다시 학습 모드로 돌아옵니다.
model.train()


미래 토큰 변경 후 prefix logits 최대 차이: 0.0


GPT1LanguageModel(
  (token_embedding): Embedding(4000, 384)
  (position_embedding): Embedding(128, 384)
  (embedding_dropout): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-11): 12 x GPT1Block(
      (attn): CausalSelfAttention(
        (qkv_proj): Linear(in_features=384, out_features=1152, bias=True)
        (out_proj): Linear(in_features=384, out_features=384, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT1MLP(
        (fc): Linear(in_features=384, out_features=1536, bias=True)
        (proj): Linear(in_features=1536, out_features=384, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
    )
  )
  (lm_head): Linear(in_features=384, out_features=4000, bias=False)
)


## 14. GPT-1 사전학습

질문과 답변을 합친 전체 토큰열에서 **다음 토큰 예측 loss**를 학습합니다.  
별도의 감정 분류, 검색 fallback, 지도학습 head는 사용하지 않습니다.

학습 셀을 다시 실행해도 이전 weight가 누적되지 않도록 모델과 optimizer를 새로 만듭니다.


In [ ]:

# GPU에서는 조금 더, CPU에서는 가볍게 학습합니다.
EPOCHS = (
    5
    if DEVICE.type == "cuda"
    else 3
)

# AdamW의 최대 학습률입니다.
LEARNING_RATE = 2.5e-4

# 작은 weight decay로 과적합을 줄입니다.
WEIGHT_DECAY = 0.01

# gradient 폭주를 막는 안전벨트입니다.
GRAD_CLIP_NORM = 1.0

# 가장 좋은 모델을 저장할 위치입니다.
CHECKPOINT_PATH = (
    DATA_DIR
    / "best_gpt1_pretrained.pt"
)

# 학습 셀 재실행 시 이전 weight를 깨끗이 지웁니다.
set_seed(SEED)

# [QUEST 4] GPT-1 모델을 새로 만듭니다.
model = GPT1LanguageModel(
    config
).to(
    DEVICE
)

# GPT 계열에서 자주 쓰는 AdamW를 준비합니다.
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    weight_decay=WEIGHT_DECAY,
)

# 전체 학습 step 수를 계산합니다.
total_steps = max(
    1,
    EPOCHS
    * len(train_loader),
)

# 초반 5%는 천천히 학습률을 올립니다.
warmup_steps = max(
    1,
    int(
        total_steps
        * 0.05
    ),
)

# warmup 뒤 cosine으로 학습률을 줄이는 함수입니다.
def learning_rate_factor(
    step: int,
) -> float:
    # 초반에는 학습률을 서서히 올립니다.
    if step < warmup_steps:
        # 0에서 1까지 부드럽게 올라갑니다.
        return (
            step + 1
        ) / warmup_steps

    # warmup 이후 진행 비율을 계산합니다.
    progress = (
        step
        - warmup_steps
    ) / max(
        1,
        total_steps
        - warmup_steps,
    )

    # 진행 비율을 안전한 범위로 자릅니다.
    progress = min(
        max(
            progress,
            0.0,
        ),
        1.0,
    )

    # cosine 곡선으로 학습률을 천천히 낮춥니다.
    return 0.5 * (
        1.0
        + math.cos(
            math.pi
            * progress
        )
    )

# step마다 학습률을 조절합니다.
scheduler = (
    torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=learning_rate_factor,
    )
)

# gradient 없이 검증 loss를 계산합니다.
@torch.no_grad()
def evaluate(
    model: GPT1LanguageModel,
    loader: DataLoader,
) -> tuple[float, float]:
    # dropout을 끄고 평가 모드로 전환합니다.
    model.eval()

    # loss 합계를 담습니다.
    total_loss = 0.0

    # 배치 개수를 셉니다.
    batch_count = 0

    # 검증 배치를 하나씩 살펴봅니다.
    for batch in tqdm(
        loader,
        desc="valid",
        leave=False,
    ):
        # 각 텐서를 학습 장치로 보냅니다.
        batch = {
            key: value.to(
                DEVICE,
                non_blocking=True,
            )
            for key, value in batch.items()
        }

        # 검증 logits와 loss를 계산합니다.
        _, loss = model(
            input_ids=batch["input_ids"],
            position_ids=batch["position_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
        )

        # 현재 loss를 합계에 더합니다.
        total_loss += float(
            loss
        )

        # 배치 수를 하나 늘립니다.
        batch_count += 1

    # 평균 검증 loss를 계산합니다.
    mean_loss = (
        total_loss
        / max(
            1,
            batch_count,
        )
    )

    # loss를 직관적인 perplexity로 바꿉니다.
    perplexity = math.exp(
        min(
            mean_loss,
            20.0,
        )
    )

    # 다시 학습 모드로 돌아옵니다.
    model.train()

    # 평균 loss와 perplexity를 반환합니다.
    return mean_loss, perplexity

# 학습 기록을 저장할 표지판입니다.
history = {
    "epoch": [],
    "train_loss": [],
    "valid_loss": [],
    "valid_ppl": [],
    "learning_rate": [],
}

# 학습 전 검증 성능을 먼저 측정합니다.
initial_valid_loss, initial_valid_ppl = evaluate(
    model,
    valid_loader,
)

# 출발점의 성능을 출력합니다.
print(
    "[학습 전] "
    f"valid_loss={initial_valid_loss:.4f} "
    f"valid_ppl={initial_valid_ppl:.2f}"
)

# 가장 좋은 검증 loss는 무한대에서 시작합니다.
best_valid_loss = float(
    "inf"
)

# 전체 step 번호를 0에서 시작합니다.
global_step = 0

# 지정한 epoch만큼 반복합니다.
for epoch in range(
    1,
    EPOCHS + 1,
):
    # 모델을 학습 모드로 둡니다.
    model.train()

    # 한 epoch의 loss 합계를 초기화합니다.
    epoch_loss_sum = 0.0

    # 한 epoch의 배치 수를 초기화합니다.
    epoch_batch_count = 0

    # 학습 진행바를 준비합니다.
    progress_bar = tqdm(
        train_loader,
        desc=f"train epoch {epoch}",
        leave=False,
    )

    # 배치를 하나씩 학습합니다.
    for batch in progress_bar:
        # 전체 step을 하나 올립니다.
        global_step += 1

        # 각 텐서를 학습 장치로 보냅니다.
        batch = {
            key: value.to(
                DEVICE,
                non_blocking=True,
            )
            for key, value in batch.items()
        }

        # 이전 gradient를 깨끗이 비웁니다.
        optimizer.zero_grad(
            set_to_none=True
        )

        # [QUEST 4] 다음 토큰 예측 loss를 계산합니다.
        _, loss = model(
            input_ids=batch["input_ids"],
            position_ids=batch["position_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
        )

        # NaN이나 inf가 나오면 즉시 멈춥니다.
        if not torch.isfinite(
            loss
        ):
            # 문제를 숨기지 않고 크게 알립니다.
            raise RuntimeError(
                f"유효하지 않은 loss가 발생했습니다: {loss}"
            )

        # loss를 뒤로 전파합니다.
        loss.backward()

        # gradient에 안전벨트를 채웁니다.
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP_NORM,
        )

        # 모델 weight를 한 걸음 업데이트합니다.
        optimizer.step()

        # 학습률도 다음 step으로 이동합니다.
        scheduler.step()

        # 현재 loss를 숫자로 꺼냅니다.
        current_loss = float(
            loss.detach()
        )

        # epoch loss 합계에 더합니다.
        epoch_loss_sum += current_loss

        # 배치 수도 하나 늘립니다.
        epoch_batch_count += 1

        # 진행바에 loss와 학습률을 표시합니다.
        progress_bar.set_postfix(
            loss=f"{current_loss:.4f}",
            lr=(
                f"{optimizer.param_groups[0]['lr']:.2e}"
            ),
        )

    # 한 epoch의 평균 학습 loss를 계산합니다.
    mean_train_loss = (
        epoch_loss_sum
        / max(
            1,
            epoch_batch_count,
        )
    )

    # 검증 loss와 perplexity를 계산합니다.
    valid_loss, valid_ppl = evaluate(
        model,
        valid_loader,
    )

    # 현재 학습률을 기록합니다.
    current_lr = optimizer.param_groups[
        0
    ]["lr"]

    # epoch 번호를 기록합니다.
    history["epoch"].append(
        epoch
    )

    # 학습 loss를 기록합니다.
    history["train_loss"].append(
        mean_train_loss
    )

    # 검증 loss를 기록합니다.
    history["valid_loss"].append(
        valid_loss
    )

    # 검증 perplexity를 기록합니다.
    history["valid_ppl"].append(
        valid_ppl
    )

    # 학습률도 함께 기록합니다.
    history["learning_rate"].append(
        current_lr
    )

    # [QUEST 4] 학습 진행 결과를 출력합니다.
    print(
        f"[epoch {epoch:02d}/{EPOCHS}] "
        f"train_loss={mean_train_loss:.4f} "
        f"valid_loss={valid_loss:.4f} "
        f"valid_ppl={valid_ppl:.2f} "
        f"lr={current_lr:.3e}"
    )

    # 검증 loss가 좋아졌는지 확인합니다.
    if valid_loss < best_valid_loss:
        # 가장 좋은 기록을 갱신합니다.
        best_valid_loss = valid_loss

        # 모델과 설정을 함께 저장합니다.
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "config": asdict(config),
                "tokenizer_path": str(TOKENIZER_PATH),
                "best_valid_loss": best_valid_loss,
                "history": history,
            },
            CHECKPOINT_PATH,
        )

        # 저장 사실을 알려 줍니다.
        print(
            "  ↳ best checkpoint 저장:",
            CHECKPOINT_PATH,
        )

# 가장 좋은 검증 loss를 출력합니다.
print(
    "학습 완료, best valid loss:",
    best_valid_loss,
)


valid:   0%|          | 0/588 [00:00<?, ?it/s]

[학습 전] valid_loss=8.4108 valid_ppl=4495.20


train epoch 1:   0%|          | 0/5288 [00:00<?, ?it/s]


## 15. 학습 진행 시각화

학습 loss와 검증 loss가 어떻게 움직였는지 그래프로 확인합니다.  
숫자는 발자국이고, 그래프는 산책 경로입니다.


In [ ]:

# 학습 기록을 표로 바꿉니다.
history_df = pd.DataFrame(
    history
)

# 기록 전체를 표로 확인합니다.
display(
    history_df
)

# 그래프 크기를 정합니다.
plt.figure(
    figsize=(8, 5)
)

# 학습 loss 선을 그립니다.
plt.plot(
    history["epoch"],
    history["train_loss"],
    marker="o",
    label="train loss",
)

# 검증 loss 선도 함께 그립니다.
plt.plot(
    history["epoch"],
    history["valid_loss"],
    marker="o",
    label="valid loss",
)

# 가로축 이름을 붙입니다.
plt.xlabel(
    "epoch"
)

# 세로축 이름을 붙입니다.
plt.ylabel(
    "cross-entropy loss"
)

# 그래프 제목을 붙입니다.
plt.title(
    "GPT-1 Pre-training Progress"
)

# 선의 이름표를 보여 줍니다.
plt.legend()

# 읽기 쉬운 격자를 살짝 깝니다.
plt.grid(
    alpha=0.3
)

# 그래프를 화면에 출력합니다.
plt.show()



## 16. 질문에 따른 답변 생성

가장 좋은 체크포인트를 불러온 뒤, 질문 prompt 다음 토큰을 한 개씩 생성합니다.

```text
<bos><speaker1>사용자 질문<turn><speaker2>
```

`<eos>`가 나오면 생성을 멈춥니다.  
출력 품질보다 **모델이 입력에 따라 정상적으로 autoregressive 생성하는지**가 평가 핵심입니다.


In [ ]:

# 저장된 가장 좋은 체크포인트를 불러옵니다.
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
)

# 저장된 weight를 모델에 적용합니다.
model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

# 생성 중 dropout을 끕니다.
model.eval()

# 반복 토큰의 점수를 낮추는 함수입니다.
def apply_repetition_penalty(
    logits: torch.Tensor,
    generated_ids: list[int],
    penalty: float = 1.2,
) -> torch.Tensor:
    # 이미 등장한 토큰만 한 번씩 확인합니다.
    for token_id in set(
        generated_ids
    ):
        # 특수 토큰은 별도 규칙으로 처리하므로 건너뜁니다.
        if token_id in {
            PAD_ID,
            UNK_ID,
            BOS_ID,
            EOS_ID,
            SPEAKER1_ID,
            SPEAKER2_ID,
            TURN_ID,
        }:
            # 다음 토큰으로 이동합니다.
            continue

        # 양수 logits는 나눠서 살짝 낮춥니다.
        if logits[
            0,
            token_id,
        ] > 0:
            # 이미 나온 토큰의 재등장을 덜 매력적으로 만듭니다.
            logits[
                0,
                token_id,
            ] /= penalty

        # 음수 logits는 곱해서 더 낮춥니다.
        else:
            # 반복 후보를 한 걸음 더 멀리 보냅니다.
            logits[
                0,
                token_id,
            ] *= penalty

    # 조정한 logits를 반환합니다.
    return logits

# top-k와 top-p 후보에서 다음 토큰을 뽑습니다.
def sample_next_token(
    logits: torch.Tensor,
    temperature: float,
    top_k: int,
    top_p: float,
) -> torch.Tensor:
    # 온도가 낮을수록 선택이 차분해집니다.
    logits = logits / max(
        temperature,
        1e-5,
    )

    # 상위 k개 후보만 남깁니다.
    if top_k > 0:
        # 실제 어휘 크기를 넘지 않게 합니다.
        k = min(
            top_k,
            logits.size(-1),
        )

        # k번째 점수를 기준선으로 잡습니다.
        threshold = torch.topk(
            logits,
            k,
        ).values[
            :,
            -1,
        ].unsqueeze(-1)

        # 기준선 아래 후보는 무대 밖으로 보냅니다.
        logits = logits.masked_fill(
            logits < threshold,
            float("-inf"),
        )

    # 현재 logits를 확률로 바꿉니다.
    probabilities = F.softmax(
        logits,
        dim=-1,
    )

    # 확률이 큰 순서대로 정렬합니다.
    sorted_probabilities, sorted_indices = torch.sort(
        probabilities,
        descending=True,
        dim=-1,
    )

    # 누적 확률을 계산합니다.
    cumulative_probabilities = torch.cumsum(
        sorted_probabilities,
        dim=-1,
    )

    # top-p 범위를 넘어선 후보를 표시합니다.
    remove_mask = (
        cumulative_probabilities
        > top_p
    )

    # 가장 높은 확률 토큰 하나는 항상 남깁니다.
    remove_mask[
        :,
        0,
    ] = False

    # top-p 밖 후보의 확률을 0으로 만듭니다.
    sorted_probabilities = sorted_probabilities.masked_fill(
        remove_mask,
        0.0,
    )

    # 남은 확률의 합이 1이 되도록 다시 맞춥니다.
    sorted_probabilities = (
        sorted_probabilities
        / sorted_probabilities.sum(
            dim=-1,
            keepdim=True,
        )
    )

    # 정렬된 후보 중 하나를 확률적으로 뽑습니다.
    sampled_position = torch.multinomial(
        sorted_probabilities,
        num_samples=1,
    )

    # 원래 vocabulary token ID로 되돌립니다.
    next_token = torch.gather(
        sorted_indices,
        dim=-1,
        index=sampled_position,
    )

    # 선택한 다음 토큰을 반환합니다.
    return next_token

# [QUEST 5] 질문에 따라 답변을 생성합니다.
@torch.no_grad()
def generate_answer(
    question: str,
    max_new_tokens: int = 50,
    temperature: float = 0.8,
    top_k: int = 30,
    top_p: float = 0.9,
) -> str:
    # 생성 시 dropout을 끕니다.
    model.eval()

    # 사용자 질문을 학습 때와 같은 방식으로 정리합니다.
    question = clean_text(
        question
    )

    # 답변 시작 지점까지 prompt를 만듭니다.
    prompt = (
        f"<bos>"
        f"<speaker1>{question}"
        f"<turn>"
        f"<speaker2>"
    )

    # prompt를 token ID로 바꿉니다.
    prompt_ids = tokenizer.encode(
        prompt
    ).ids

    # 너무 긴 prompt는 최근 문맥만 남깁니다.
    prompt_ids = prompt_ids[
        -config.block_size:
    ]

    # token ID를 배치 크기 1의 텐서로 만듭니다.
    input_ids = torch.tensor(
        prompt_ids,
        dtype=torch.long,
        device=DEVICE,
    ).unsqueeze(
        0
    )

    # 새로 생성된 답변 토큰을 담습니다.
    generated_ids: list[int] = []

    # 최대 토큰 수만큼 한 토큰씩 생성합니다.
    for step in range(
        max_new_tokens
    ):
        # 현재 문맥 창에 들어가는 토큰만 사용합니다.
        context_ids = input_ids[
            :,
            -config.block_size:
        ]

        # 현재 문맥 길이를 구합니다.
        context_length = context_ids.size(
            1
        )

        # [QUEST 3] 현재 문맥 길이에 맞는 위치 번호를 만듭니다.
        position_ids = torch.arange(
            context_length,
            dtype=torch.long,
            device=DEVICE,
        ).unsqueeze(
            0
        )

        # 실제 토큰만 있으므로 mask는 모두 1입니다.
        attention_mask = torch.ones_like(
            context_ids
        )

        # 현재 문맥에서 다음 토큰 logits를 계산합니다.
        logits, _ = model(
            input_ids=context_ids,
            position_ids=position_ids,
            attention_mask=attention_mask,
        )

        # 마지막 위치의 vocabulary 점수만 꺼냅니다.
        next_logits = logits[
            :,
            -1,
            :,
        ].clone()

        # 답변 중간에 다시 나오면 곤란한 토큰입니다.
        forbidden_ids = [
            PAD_ID,
            UNK_ID,
            BOS_ID,
            SPEAKER1_ID,
            SPEAKER2_ID,
            TURN_ID,
        ]

        # 구조용 토큰의 점수를 막습니다.
        next_logits[
            :,
            forbidden_ids,
        ] = float(
            "-inf"
        )

        # 첫 두 토큰 안에는 너무 빨리 끝나지 않게 합니다.
        if step < 2:
            # EOS를 잠시 대기실에 둡니다.
            next_logits[
                :,
                EOS_ID,
            ] = float(
                "-inf"
            )

        # 반복 토큰의 매력을 낮춥니다.
        next_logits = apply_repetition_penalty(
            next_logits,
            generated_ids,
            penalty=1.2,
        )

        # top-k와 top-p로 다음 토큰을 고릅니다.
        next_token = sample_next_token(
            next_logits,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
        )

        # 선택된 token ID를 숫자로 꺼냅니다.
        next_token_id = int(
            next_token.item()
        )

        # EOS가 나오면 문장 열차를 정차합니다.
        if next_token_id == EOS_ID:
            # 생성 반복을 끝냅니다.
            break

        # 답변 토큰 목록에 추가합니다.
        generated_ids.append(
            next_token_id
        )

        # 다음 step의 입력 뒤에 새 토큰을 붙입니다.
        input_ids = torch.cat(
            [
                input_ids,
                next_token,
            ],
            dim=1,
        )

    # 생성한 답변 토큰만 문장으로 복원합니다.
    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    )

    # 양끝 공백을 정리해 반환합니다.
    return answer.strip()



## 17. 생성 결과 확인

여러 질문을 넣어 입력에 따라 출력이 생성되는지 확인합니다.  
짧은 학습에서는 문장이 어색할 수 있지만, 과제 기준은 생성 파이프라인의 정상 동작 여부입니다.


In [ ]:

# 재현 가능한 샘플링을 위해 씨앗을 다시 고정합니다.
set_seed(SEED)

# 서로 다른 성격의 질문을 준비합니다.
sample_questions = [
    "안녕, 오늘 기분은 어때?",
    "공부가 잘 안 될 때는 어떻게 하면 좋을까?",
    "주말에 할 일을 추천해 줘.",
    "너는 누구야?",
    "비가 오는 날에는 무엇을 하면 좋을까?",
]

# 결과를 표로 모을 빈 리스트입니다.
generation_results = []

# 질문을 하나씩 모델에 건넵니다.
for question in sample_questions:
    # [QUEST 5] 질문에 따른 답변을 생성합니다.
    answer = generate_answer(
        question,
        max_new_tokens=50,
        temperature=0.8,
        top_k=30,
        top_p=0.9,
    )

    # 질문과 답변을 보기 좋게 출력합니다.
    print("=" * 70)

    # 입력 질문을 출력합니다.
    print(
        "Q:",
        question,
    )

    # 생성 답변을 출력합니다.
    print(
        "A:",
        answer
        if answer
        else "(모델이 빈 답변을 생성했습니다.)",
    )

    # 표에 넣을 결과를 저장합니다.
    generation_results.append(
        {
            "question": question,
            "generated_answer": answer,
        }
    )

# 질문별 결과를 DataFrame으로 만듭니다.
generation_df = pd.DataFrame(
    generation_results
)

# 제출 결과를 표로 보여 줍니다.
display(
    generation_df
)

# 생성 과정이 모두 문자열로 끝났는지 확인합니다.
assert all(
    isinstance(
        answer,
        str,
    )
    for answer in generation_df[
        "generated_answer"
    ]
)

# 정상 종료 메시지를 출력합니다.
print(
    "입력별 autoregressive 생성이 정상 종료되었습니다."
)



## 18. QUEST 평가 기준 자동 점검

마지막으로 평가 기준 1~5가 코드에 실제로 들어 있는지 표로 확인합니다.


In [ ]:

# 평가 항목과 코드 근거를 표로 정리합니다.
quest_checklist = pd.DataFrame(
    [
        {
            "평가 기준": "1. Transformer 대비 변경",
            "코드 근거": (
                "첫 Markdown 구조 블록, "
                "CausalSelfAttention, GPT1Block"
            ),
            "확인": True,
        },
        {
            "평가 기준": "2. Decoder 생성 전처리",
            "코드 근거": (
                "serialize_dialogue(), "
                "GPT1PretrainDataset"
            ),
            "확인": True,
        },
        {
            "평가 기준": "3. 위치 정보 입력",
            "코드 근거": (
                "position_ids, "
                "position_embedding, "
                "token + position"
            ),
            "확인": True,
        },
        {
            "평가 기준": "4. GPT-1 구성·학습",
            "코드 근거": (
                "print(model), 12 GPT blocks, "
                "history, training curve"
            ),
            "확인": True,
        },
        {
            "평가 기준": "5. 입력별 출력 생성",
            "코드 근거": (
                "generate_answer(), "
                "generation_df"
            ),
            "확인": True,
        },
    ]
)

# 체크리스트를 화면에 보여 줍니다.
display(
    quest_checklist
)

# 모든 항목이 True인지 확인합니다.
assert quest_checklist[
    "확인"
].all()

# 최종 통과 메시지를 출력합니다.
print(
    "QUEST 평가 기준 1~5 코드 점검 완료"
)



# 회고

## 1. 무엇을 바꾸었는가

처음 파일은 질문을 Encoder에 넣고 답변을 Decoder에서 생성하는 일반 Transformer 챗봇이었습니다.  
이번 과제의 핵심은 GPT-1이므로 Encoder와 Cross-Attention을 제거하고, **Causal Self-Attention만 사용하는 Decoder-only 구조**로 다시 작성했습니다.

질문과 답변도 따로 처리하지 않고 다음처럼 하나의 토큰열로 묶었습니다.

```text
<bos><speaker1>질문<turn><speaker2>답변<eos>
```

모델은 이 토큰열에서 앞 토큰을 보고 다음 토큰을 맞히는 방식으로 사전학습합니다.

## 2. 가장 중요하게 배운 점

GPT는 단순히 Transformer의 Decoder 클래스만 가져오는 모델이 아니었습니다.  
다음 요소가 함께 맞물려야 GPT다운 입력과 학습이 완성됩니다.

- 미래 토큰을 보지 못하게 하는 causal mask
- Token Embedding에 더하는 Learned Position Embedding
- 질문과 답변을 하나로 연결한 causal language-model 데이터
- 한 칸 이동한 다음 토큰 정답
- Encoder와 Cross-Attention이 없는 Decoder-only 블록
- 입력 embedding과 출력 vocabulary weight 공유

특히 Transformer는 토큰 순서를 스스로 알 수 없기 때문에 위치 정보를 따로 넣어야 한다는 점이 인상 깊었습니다. 위치 ID가 작은 번호표라면 위치 임베딩은 그 번호표를 모델이 이해할 수 있는 좌석 안내도로 바꾸는 과정이라고 생각했습니다.

## 3. 어려웠던 점

가장 어려운 부분은 “챗봇 학습”과 “GPT 사전학습”을 구분하는 일이었습니다.  
검색 기반 fallback이나 답변 영역만 학습하는 방식은 챗봇 품질을 높이는 데 도움이 될 수 있지만, 이번 평가 기준은 GPT-1의 사전학습 구조를 확인하는 것이므로 제외했습니다.

또한 실제 GPT-1 논문 규모는 약 1억 개 이상의 파라미터를 사용하기 때문에 수업 환경에서 그대로 학습하면 시간이 오래 걸립니다. 이번 노트북은 12개 Decoder 블록은 유지하되 hidden 차원과 문맥 길이를 줄인 실습 설정을 기본값으로 사용했습니다. GPU 환경과 시간이 충분하면 `MODEL_PRESET = "paper"`로 변경할 수 있습니다.

## 4. 결과를 어떻게 해석했는가

학습 loss와 validation loss가 내려가면 모델이 다음 토큰의 패턴을 배우고 있다는 뜻입니다.  
하지만 짧은 데이터와 짧은 학습만으로는 자연스러운 답변이 나오지 않을 수 있습니다.

이번 평가에서는 답변 품질보다 다음 과정이 정상적으로 이어지는지가 더 중요합니다.

```text
질문 입력
→ 특수 토큰이 포함된 prompt 생성
→ token embedding + position embedding
→ causal decoder blocks
→ 다음 토큰 logits
→ autoregressive 답변 생성
```

## 5. 다음에 개선하고 싶은 점

앞으로는 다음 내용을 적용해 보고 싶습니다.

1. 더 크고 다양한 한국어 말뭉치로 먼저 사전학습하기
2. 사전학습 후 챗봇 데이터로 별도 fine-tuning 진행하기
3. validation perplexity와 생성 품질을 함께 평가하기
4. beam search, top-p, repetition penalty를 조건별로 비교하기
5. 학습률, batch size, 문맥 길이를 체계적으로 실험하기

이번 작업을 통해 GPT-1은 단순히 “Decoder를 사용하는 Transformer”가 아니라, **입력 구성·위치 정보·causal mask·다음 토큰 학습이 하나로 연결된 생성 모델**이라는 점을 구체적으로 이해할 수 있었습니다.
